# HPC Mismatch Check
Collect only the NCDE/RL config and shape info needed to diagnose the state-dimension mismatch.

In [ ]:
import os
import yaml
import numpy as np
import torch

# Config names (no .yaml)
ncde_config = "ncde_config_continuous"
rl_config = "iqn_continuous_mimic"

# Load configs
ncde_params = yaml.safe_load(open(os.path.join("configs", f"{ncde_config}.yaml")))
rl_params = yaml.safe_load(open(os.path.join("configs", f"{rl_config}.yaml")))

# Resolve paths
encode_dir = ncde_params.get("encode_output_dir", ncde_params["data_dir"])
ncde_output_dir = ncde_params["output_dir"]
rl_data_dir = rl_params["data_dir"]
ckpt_dir = rl_params["checkpoint_fname"]

print("NCDE encode_output_dir:", encode_dir)
print("NCDE output_dir:", ncde_output_dir)
print("RL data_dir:", rl_data_dir)
print("RL checkpoint_fname:", ckpt_dir)

# Encoded data dims (from RL data_dir)
enc_path = os.path.join(rl_data_dir, "encoded_train.npz")
enc = np.load(enc_path, allow_pickle=True)
enc_state_dim = enc["states"].shape[-1]
print("encoded_train state_dim:", enc_state_dim)

# NCDE hidden_dim from best_model.pt
ncde_ckpt_path = os.path.join(ncde_output_dir, "best_model.pt")
ncde_ckpt = torch.load(ncde_ckpt_path, map_location="cpu", weights_only=False)
ncde_hidden_dim = ncde_ckpt["hyperparameters"]["hidden_dim"]
print("NCDE hidden_dim:", ncde_hidden_dim)

# RL checkpoint expected state_dim
ckpt_path = os.path.join(ckpt_dir, "best_q_parametersnegative.pt")
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
head_w = ckpt["rl_network_state_dict"]["head.weight"]
action_dim = rl_params.get("action_dim", 2)
ckpt_state_dim = head_w.shape[1] - action_dim
print("checkpoint head.weight shape:", tuple(head_w.shape))
print("checkpoint inferred state_dim:", ckpt_state_dim)

# Report mismatch summary
print("\nMismatch summary:")
print("- encoded state_dim == NCDE hidden_dim:", enc_state_dim == ncde_hidden_dim)
print("- encoded state_dim == checkpoint state_dim:", enc_state_dim == ckpt_state_dim)